# Exercise 02 — Interactive Maps with Folium

In the previous exercises, we used Pandas to organise datasets and graphs to represent numerical patterns. **Maps allow us to explore where things happen.** An interactive map can reveal clusters, gaps and spatial relationships that are difficult to recognise in a table.

**Folium** is a Python library that creates interactive maps using [Leaflet](https://leafletjs.com/), a JavaScript mapping library. We build a map in Python, view it inside Jupyter and can save it as an HTML file to share or open in a browser.

![Folium logo](https://python-visualization.github.io/folium/latest/_images/folium_logo.png)

This exercise begins with a simple map of London. We will change its background style, add markers and then plot the locations of photographs from our Flickr dataset.

You can find more examples in the [Folium documentation](https://python-visualization.github.io/folium/).

## Installation

If Folium is not already installed, run this in a notebook cell:

```python
%pip install folium pandas
```

## Creating a map

A map needs two starting parameters:

- **`location`** — the centre of the map, written as **[latitude, longitude]**. Latitude describes north–south position and longitude describes east–west position.
- **`zoom_start`** — the initial zoom level. A larger number means a closer view and more detail.

![Diagram explaining latitude, longitude, map centre and zoom](assets/map-centre-and-zoom.svg)

The centre below is in London, near the River Thames. Change either coordinate or the zoom level to see how the view changes.

### A first Folium map

Import the Folium library, create a `folium.Map()` object and display it by writing the variable name at the end of the cell.

In [ ]:
import folium

# Coordinates are always [latitude, longitude] in Folium
centre = [51.505650528792636, -0.08408494665571324]

map_london = folium.Map(
    location=centre,
    zoom_start=16,
    control_scale=True
)

map_london

### If the map does not display

Folium produces interactive HTML. If the map is blank **inside VS Code**, the notebook's HTML output may be blocked. Open the Command Palette (**Ctrl+Shift+P**) and look for **Notebook: Trust Notebook**, if available. Only trust files from sources you recognise.

If the markers are visible but the background is missing, the **map tiles** may not have loaded. Check your internet connection and the tile provider. This tutorial now uses OpenStreetMap, which does not require a CARTO API key.

**Test in a browser:** use `map_london.save("test_map.html")` and open the resulting HTML file in Chrome or another browser. If it works there but not in the notebook, investigate the notebook renderer. An internet connection is still needed for live background tiles.

### Changing the basemap

The `tiles` parameter controls the **background map**, separately from your markers and data layers. We will use **OpenStreetMap** throughout this tutorial because it works without an API key.

Older Folium tutorials often use the attractive light **CARTO Positron** or dark **CARTO Dark Matter** basemaps. **CARTO now requires an API key for these styles.** Using `tiles="CartoDB dark_matter"` without configuring a key can result in missing tiles or an API-key notice. See [CARTO's basemap key information](https://carto.com/basemaps/apikey/) and the [Folium guide to tile layers](https://python-visualization.github.io/folium/latest/user_guide/raster_layers/tiles.html).

For now, try changing the map's **zoom level** while keeping the same OpenStreetMap tiles.

In [ ]:
# OpenStreetMap works without a CARTO API key
map_london = folium.Map(
    location=centre,
    tiles="OpenStreetMap",
    zoom_start=14,
    control_scale=True
)

map_london

### Adding markers

A map becomes useful when we add **data points**. Folium's `CircleMarker` draws a circle at a specified location.

- **`location`** is a coordinate in `[latitude, longitude]` order.
- **`radius`** is measured in **screen pixels**: the marker remains the same size when you zoom. In contrast, `folium.Circle` uses a radius in **metres**.
- **`color`**, **`fill_color`**, **`stroke`** and **`fill_opacity`** control its appearance.
- **`tooltip`** displays a short label when you hover over a point.
- **`popup`** shows information when you click a point.

This example marks Covent Garden. The final ` .add_to(map_london)` call attaches the marker to the map. See the [CircleMarker documentation](https://python-visualization.github.io/folium/latest/user_guide/vector_layers/circle_and_circle_marker.html).

In [ ]:
# Add a labelled point for Covent Garden
folium.CircleMarker(
    location=[51.51167312861235, -0.12335130551220098],
    radius=6,  # pixels, not metres
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_color="cornflowerblue",
    fill_opacity=0.7,
    tooltip="Covent Garden",  # hover
    popup="A point in central London"  # click
).add_to(map_london)

map_london

### Save the interactive map as HTML

Unlike a static screenshot, an HTML export retains the ability to pan, zoom and click markers. Open the saved file in your web browser to test it independently of Jupyter. The background tiles still need an internet connection.

If the map is blank in the notebook but works in the browser, check notebook trust and HTML rendering. If the points appear but streets do not, check which tile provider is being requested.

In [ ]:
# Save the map with its marker as an interactive HTML file
map_london.save("map_with_marker.html")

## Mapping Flickr Photographs

Now we can connect the mapping techniques to the **Flickr dataset** used in Day 2. Each photograph has a latitude (`lat`) and longitude (`lon`), allowing us to represent it as a point on the map.

We will first display **400 photographs**. This is enough to see patterns without creating the extremely large notebook output that thousands of individual markers can produce. The data preparation is important:

1. Load the dataset with **Pandas**.
2. Convert the coordinate columns into numbers with `pd.to_numeric()`. Invalid text becomes a missing value.
3. Remove incomplete or impossible coordinates. Latitude must be between −90 and 90; longitude must be between −180 and 180.
4. Select a manageable sample and loop through its coordinates to create a `CircleMarker` for every photograph.

For simplicity, the map starts centred on the same London location as before. A coordinate with `0, 0` can technically be valid, so we do not automatically discard it; inspect your study area before deciding whether a location is plausible.

In [ ]:
# Load the geotagged Flickr dataset
import pandas as pd
import folium

flickr_df = pd.read_csv("assets/Flickr.csv")

# Convert both coordinate columns to numeric values
flickr_df["lat"] = pd.to_numeric(flickr_df["lat"], errors="coerce")
flickr_df["lon"] = pd.to_numeric(flickr_df["lon"], errors="coerce")

# Keep only rows with complete coordinates in valid ranges
valid_locations = flickr_df.dropna(subset=["lat", "lon"]).copy()
valid_locations = valid_locations[
    valid_locations["lat"].between(-90, 90)
    & valid_locations["lon"].between(-180, 180)
]

print(f"Records in dataset: {len(flickr_df):,}")
print(f"Records with valid coordinates: {len(valid_locations):,}")

# Start with a manageable number of photographs
sample = valid_locations.head(400)

flickr_map = folium.Map(
    location=centre,
    tiles="OpenStreetMap",
    zoom_start=13,
    control_scale=True
)

# Each row supplies one latitude and longitude pair
for latitude, longitude in sample[["lat", "lon"]].itertuples(index=False, name=None):
    folium.CircleMarker(
        location=[latitude, longitude],
        radius=2,
        stroke=False,
        fill=True,
        fill_color="cornflowerblue",
        fill_opacity=0.6
    ).add_to(flickr_map)

flickr_map

### Save and inspect your results

The map shows the **locations recorded in the Flickr metadata**, not necessarily where the photographs were taken with metre-level precision. Look for spatial concentrations and gaps. Are they patterns of urban activity, patterns of photography, or effects of how the photographs were collected?

The map is interactive, but a large number of individual circles can make its HTML file heavy. We can export the current version and open it in a browser: 

In [ ]:
# Open the exported map in a web browser
flickr_map.save("map_London.html")

## Optional — Marker Clustering

When a map contains many photographs, nearby markers may overlap and become difficult to interpret. **MarkerCluster** groups points at lower zoom levels; as you zoom in, the groups divide into smaller clusters and individual markers.

This is useful for **navigating a dense map**, but it is not a statistical density analysis: the clusters depend on the map's zoom and screen position. The numbers shown on the clusters count markers, not necessarily distinct photographers or events.

The example below uses up to **1,500** photographs. We use ordinary `folium.Marker` objects because this works naturally with the clustering plugin. Keep the number modest while experimenting, and export the result to HTML instead of embedding a very large map output in the notebook.

In [ ]:
from folium.plugins import MarkerCluster

cluster_map = folium.Map(
    location=centre,
    tiles="OpenStreetMap",
    zoom_start=12
)

cluster_layer = MarkerCluster().add_to(cluster_map)

# Group nearby markers; use more records only when necessary
cluster_sample = valid_locations.head(1500)

for latitude, longitude in cluster_sample[["lat", "lon"]].itertuples(index=False, name=None):
    folium.Marker(
        location=[latitude, longitude]
    ).add_to(cluster_layer)

cluster_map.save("map_London_clusters.html")
print(f"Saved a clustered map of {len(cluster_sample):,} photographs.")

### Individual points or clusters?

A **CircleMarker map** is a good choice when you want to control the visual appearance of each point and examine smaller datasets. **MarkerCluster** makes a larger dataset easier to navigate, but it changes how the points are presented as you zoom. Neither method, by itself, establishes the intensity of activity in a city.

Try comparing the two HTML files in your browser. Do the same spatial patterns remain visible? Consider whether a grid, heatmap or statistical aggregation would provide a more meaningful representation for your research question. These questions will become relevant again in our Grasshopper grid-sampling exercises.